Чтение файла с названиями и сайтами

In [1]:
import pandas as pd
df=pd.read_csv('../Тест на 3 строительные компании_2.csv', delimiter=';')
# print(df)
names=df.iloc[:, 0:4]
sites=df.iloc[:, 4:]
print('names:\n', names)
print('\n\n')
print('sites:\n', sites)

names:
                                        original_name  \
0          ООО Калининградская Строительная Компания   
1                     ООО НИС.ЖИЛИЩНОЕ СТРОИТЕЛЬСТВО   
2  ФОНД ЖИЛИЩНОЕ И СОЦИАЛЬНОЕ СТРОИТЕЛЬСТВО КАЛИН...   

                 short_name_1 short_name_2  \
0                      КСК-39    КСК Строй   
1  Нис.жилищное строительство       Эрбиай   
2                        ФЖСС    НО «ФЖСС»   

                                   short_name_3  
0                              кск-недвижимость  
1                                           RBI  
2  Группа компаний ФЖСС Калининградской области  



sites:
       site_1                 site_2     site_3       site_4  \
0     kgd.ru  www.newkaliningrad.ru  ruwest.ru     klops.ru   
1  www.dp.ru        www.fontanka.ru  47news.ru  online47.ru   
2     kgd.ru  www.newkaliningrad.ru  ruwest.ru     klops.ru   

                site_5  
0  gov39.ru/press/news  
1              ivbg.ru  
2  gov39.ru/press/news  


Общие функции

In [2]:
import os
import json
import re
import sys
from datetime import datetime

In [3]:
def debug_json_error(s, e):
    lines = s.splitlines()
    error_line = e.lineno - 1

    print("\n--- JSON ERROR CONTEXT ---")
    print(lines[error_line])
    print(" " * (e.colno - 1) + "^")

    print(f"\nError: {e.msg}")
    print(f"Line: {e.lineno}, Column: {e.colno}")

def debug_json_error(s, e):
    lines = s.splitlines()
    
    error_line = e.lineno - 1  # индекс
    start = max(0, error_line - 2)
    end = min(len(lines), error_line + 3)

    print("\n--- JSON ERROR CONTEXT ---")
    for i in range(start, end):
        prefix = ">> " if i == error_line else "   "
        print(f"{prefix}{i+1}: {lines[i]}")

    print(f"\nError: {e.msg}")
    print(f"Line: {e.lineno}, Column: {e.colno}, Char: {e.pos}")

In [ ]:
from json_repair import repair_json

def extract_json(mess, name_of_company, site):
    evaluation_json=None
    try:
        # Try to find JSON within triple backticks
        json_match = re.search(r'```(?:json)?\s*({[\s\S]*?})\s*```', mess, re.DOTALL)
        if json_match:
            evaluation_json = json.loads(json_match.group(1))
        else:
                # Try to find any JSON-like structure
            json_match = re.search(r'({[\s\S]*?"news"[\s\S]*?})', mess, re.DOTALL)
            if json_match:
                evaluation_json = json.loads(json_match.group(1))
            else:
                # Last resort: treat the entire response as JSON
                evaluation_json = json.loads(mess)

        print(evaluation_json)
        
    except:
        try:
            fixed = repair_json(mess)
            evaluation_json = json.loads(fixed)
        except Exception as e:
            print(f"Error parsing JSON response: {e}")
            debug_json_error(mess, e)
            if evaluation_json is None: 
                evaluation_json = {"news": 
                        [{
                            "CompanyName": str(name_of_company),
                            "NewsTitle": "NoNews",
                            "NewsDate": "NoNews",
                            "NewsSource": site,
                            "NewsURL": "NoNews",
                            "NewsText": "NoNews",
                        }]
                }
    return evaluation_json

{'new': [{'CompanyName': 'ООО Калининградская Строительная Компания', 'NewsTitle': 'Калининградская Строительная Компания расширяет проекты в регионе', 'NewsDate': '2025-03-15', 'NewsSource': 'kgd.ru', 'NewsURL': 'https://kgd.ru/news/2025/03/15/ksk-rasjirenie-proektov/', 'NewsText': 'В Калининграде продолжается активное развитие строительной отрасли. ООО Калининградская Строительная Компания объявила о запуске нового жилого комплекса в центре города. Проект включает в себя современные квартиры и инфраструктуру. Эксперты отмечают, что это поможет решить проблему дефицита жилья. Кроме того, компания планирует инвестировать в экологические технологии. ООО Калининградская Строительная Компания, известная своими проектами, уже реализовала несколько объектов в области. Жители ожидают, что это принесет положительные изменения в городской среде. Подробности можно найти на официальном сайте компании.'}, {'CompanyName': 'КСК-39', 'NewsTitle': 'КСК-39 инвестирует в инфраструктуру Калининграда', '

In [6]:
def completing_table(evaluation_json, final_table, name=None):
    for news in evaluation_json['news']:
        print(news)
        data=[]
        data.append(news['CompanyName']) #Полное название компании
        data.append(news['NewsTitle']) #Заголовок
        data.append(news['NewsDate']) #Ссылка
        data.append(news['NewsSource']) #Дата публикации
        data.append(news['NewsURL']) # Ссылка на новость 
        data.append(news['NewsText']) #Полный текст
        data.append(name) #Название компании
            # print(data)
        new_row=pd.DataFrame([data], columns=['CompanyName', 'NewsTitle', 'NewsDate', 'NewsSource', 'NewsURL', 'NewsText', 'VarName'])
            # print(new_row)
        final_table=pd.concat([final_table, new_row], axis=0, ignore_index=True)
        print(final_table)
    return final_table

In [16]:
def prompt_formation(company_name, names, site, year=2025):
    names_str = ', '.join(f'"{name}"' for name in names)
    prompt=f"""Найди ВСЕ новости, в которых упоминается компания "{company_name}" на сайте "{site}" за {year} год. Среди возможных написаний названий компании также рассмотри без учета регистра: {names_str}.
                В ответе выведи ТОЛЬКО json-объект с ТОЧНОЙ структурой:
                ```json
                    {{
                        'news':{{
                            'CompanyName': {{company name}},
                            'NewsTitle': {{title}},
                            'NewsDate': {{date}},
                            'NewsSource': {{domain where was found the article}},
                            'NewsURL': {{full url of the article}},
                            'NewsText': {{FULL text of article}},
                        }}
                    }}
                ```
                ВАЖНО:
                1. Найди ВСЕ новости по компании и её альтернативному написанию на сайте
                2. Новости должны быть строго за {year} год
                3. 'NewsText' должен содержать ПОЛНЫЙ текст статьи
                4. Все новости и ссылки на них должны быть РЕАЛЬНЫМИ
                5. Выведи ВСЕ найденные новости
                6. Чтобы добавить новость в список достаточно просто упоминания названия в ней"""

    return prompt

In [8]:
def result_txt(model, site, mess, p_token=None, c_token=None, t_token=None):
    result=f"РЕЗУЛЬТАТЫ РАБОТЫ ПОИСКОВИКА НОВОСТЕЙ НА ОСНОВЕ {model.upper()}\n\n"
    result+=f"Вывод модели {mess} для сайта {site}\n"
    if p_token:
        result+="\n--- TOTAL TOKEN USAGE ---\n"
        result+=f"  Prompt tokens: {p_token}\n"
        result+=f"  Completion tokens: {c_token}\n"
        result+=f"  Total tokens: {t_token}\n"

    timestamp = datetime.now().strftime('%Y-%m-%d-%H-%M')
    site=site.split("/")[0]
    with open(f'../Альтернативы/output_{model}_{site}_{timestamp}.txt', 'w') as file:
        file.write(result)
# result_txt('test', 'test')

2. GROK

In [ ]:
key=''

# import asyncio
from typing import Literal, Sequence, List
import base64
import mimetypes
from pydantic import BaseModel, Field
import chardet

from xai_sdk import AsyncClient
from xai_sdk.chat import system, tool, tool_result, user
from xai_sdk.search import SearchParameters

async def parser_grok(client: AsyncClient, company_name, names, site, year=2025, model="grok-3-mini") -> None:
    total_usage_info = {}

    prompt=prompt_formation(company_name, names, site, year)
    print('Запрос:', prompt)

    chat = client.chat.create(
            model=model,
            search_parameters=SearchParameters(mode="on", max_search_results=30),
            messages=[
                user(prompt), 
            ],
        )

    chat = client.chat.create(
            model=model,
            messages=[
                user(prompt), 
            ],
        )

    response = await chat.sample()
    chat.append(response)

    if response.content:
        print("Grok: ", end="", flush=True)
        print(response.content, end="", flush=True)

    if hasattr(response, 'usage') and response.usage:
            usage_info = {
            'prompt_tokens': response.usage.prompt_tokens,
            'completion_tokens': response.usage.completion_tokens,
            'total_tokens': response.usage.total_tokens
            }
            if not total_usage_info:
                total_usage_info = {
                'prompt_tokens': response.usage.prompt_tokens,
                'completion_tokens': response.usage.completion_tokens,
                'total_tokens': response.usage.total_tokens
                }
            else:
                total_usage_info = {
                'prompt_tokens': total_usage_info['prompt_tokens']+response.usage.prompt_tokens,
                'completion_tokens': total_usage_info['completion_tokens']+response.usage.completion_tokens,
                'total_tokens': total_usage_info['total_tokens']+response.usage.total_tokens
                }
        
    if usage_info:
            print("\n--- TOKEN USAGE ---\n")
            print(f"  Prompt tokens: {usage_info['prompt_tokens']}\n")
            print(f"  Completion tokens: {usage_info['completion_tokens']}\n")
            print(f"  Total tokens: {usage_info['total_tokens']}\n")

    result_txt("Grok", site, response.content, total_usage_info['prompt_tokens'], total_usage_info['completion_tokens'], total_usage_info['total_tokens'])
            
    return response.content 

In [10]:
client = AsyncClient(api_key=key)
model="grok-4-fast-reasoning"
NW_news=pd.DataFrame([], columns=['CompanyName', 'NewsTitle', 'NewsDate', 'NewsSource', 'NewsURL', 'NewsText', 'VarName'])

for c in range(len(df)):
    company_names=names.iloc[c, :]
    company_sites=sites.iloc[c, :]
    company_name=company_names.iloc[0]
    print("Company:", company_name)
    for s in company_sites:
        mess=await parser_grok(client, company_name, list(company_names), s)
        # print(mess)
        evaluation_json=extract_json(mess, company_name, s)
        NW_news=completing_table(evaluation_json, NW_news)

timestamp = datetime.now().strftime('%Y-%m-%d')
NW_news.to_csv(f"../Альтернативы/alternative_results/news_test_{model.replace('-', '_')}_{timestamp}.csv")

Company: ООО Калининградская Строительная Компания
Запрос: Найди ВСЕ новости, в которых упоминается компания "ООО Калининградская Строительная Компания" на сайте "kgd.ru" за 2025 год. Среди возможных написаний названий компании также рассмотри без учета регистра: ООО Калининградская Строительная Компания, КСК-39, КСК Строй, кск-недвижимость.
                В ответе выведи ТОЛЬКО json-объект с ТОЧНОЙ структурой:
                ```json
                    {
                        'news':{
                            'CompanyName': {company name},
                            'NewsTitle': {title},
                            'NewsDate': {date},
                            'NewsSource': {domain where was found the article},
                            'NewsURL': {full url of the article},
                            'NewsText': {FULL text of article},
                        }
                    }
                ```
                ВАЖНО:
                1. Найди ВСЕ новости по компани

3. API OpenAI (Responses API)

In [11]:
import openai
import pandas as pd
import time
import sys
import os
import re
from pathlib import Path
from io import StringIO

In [12]:
def parser_chatgpt(key, company_name, names, site, year=2025, model="gpt-5.4"):
    total_usage_info = {}
    
    openai.api_key = key
    prompt = prompt_formation(company_name, names, site, year)
    try:
        response = openai.responses.create(
            model=model,
            tools=[{"type": "web_search"}],
            input=prompt,
            temperature=0.3,
            max_output_tokens=4000
        )
        # Извлекаем текст из последнего сообщения (output)
        if response.output and response.output[-1].type == "message":
            for content in response.output[-1].content:
                if content.type == "output_text":
                    response_text=content.text
        # Если не нашли, вернём строковое представление всего ответа (для отладки)
        response_text=str(response)
    except Exception as e:
        print(f"Ошибка при запросе: {e}")
    print(response_text)
    print(response.output)
    print(response.output[-1])
    print(response.output[-1].content)
    print(response.output[-1].content[0].text)
    response_text=response.output[-1].content[0].text
    
    result_txt("ChatGPT", site, response_text)
    return response_text

In [ ]:
OPENAI_API_KEY = ""
model = "gpt-5.4"                # Ваша модель (должна поддерживать web_search)
NW_news=pd.DataFrame([], columns=['CompanyName', 'NewsTitle', 'NewsDate', 'NewsSource', 'NewsURL', 'NewsText', 'VarName'])

for c in range(len(df)):
    company_names=names.iloc[c, :]
    company_sites=sites.iloc[c, :]
    company_name=company_names.iloc[0]
    print("\nCompany:", company_name)
    for s in company_sites:
        mess=parser_chatgpt(OPENAI_API_KEY, company_name, list(company_names), s)
        # print(mess)
        evaluation_json=extract_json(mess, company_name, s)
        # print("evaluation_json:\n", evaluation_json, '\n\n')
        NW_news=completing_table(evaluation_json, NW_news)

timestamp = datetime.now().strftime('%Y-%m-%d')
NW_news.to_csv(f"../Альтернативы/alternative_results/news_test_{model.replace('-', '_')}_{timestamp}.csv")


Company: ООО Калининградская Строительная Компания
Response(id='resp_01d9cf5a246214f50069c64ef406a4819683d243472379b898', created_at=1774604020.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-2026-03-05', object='response', output=[ResponseFunctionWebSearch(id='ws_01d9cf5a246214f50069c64ef48b6c81969e5a575041022137', action=ActionSearch(query='site:kgd.ru 2025 "ООО Калининградская Строительная Компания"', type='search', queries=['site:kgd.ru 2025 "ООО Калининградская Строительная Компания"', 'site:kgd.ru 2025 "КСК-39"', 'site:kgd.ru 2025 "КСК Строй"', 'site:kgd.ru 2025 "кск-недвижимость"'], sources=None), status='completed', type='web_search_call'), ResponseFunctionWebSearch(id='ws_01d9cf5a246214f50069c64ef60ad88196bd7e771ac575fd17', action=ActionSearch(query='site:kgd.ru 2025 kgd.ru КСК 2025', type='search', queries=['site:kgd.ru 2025 kgd.ru КСК 2025', 'site:kgd.ru 2025 kgd.ru "КСК"', 'site:kgd.ru 2025 kgd.ru "КСК-39" OR "КСК Строй" OR "кск-недви